In [ ]:
import numpy as np
import serial
import time
import os

# --- MEKF CONSTANTS & UTILITIES ---
H_MAT = np.vstack([np.zeros((1, 3)), np.eye(3)])
T_MAT = np.block([[1, np.zeros((1, 3))], [np.zeros((3, 1)), -np.eye(3)]])

def hat(v):
    return np.array([[0, -v[2], v[1]], [v[2], 0, -v[0]], [-v[1], v[0], 0]])

def L(q):
    s, v = q[0], q[1:]
    m = np.zeros((4, 4))
    m[0, 0], m[0, 1:], m[1:, 0] = s, -v, v
    m[1:, 1:] = s * np.eye(3) + hat(v)
    return m

def R(q):
    s, v = q[0], q[1:]
    m = np.zeros((4, 4))
    m[0, 0], m[0, 1:], m[1:, 0] = s, -v, v
    m[1:, 1:] = s * np.eye(3) - hat(v)
    return m

def G(q):
    return L(q) @ H_MAT

def Q_mat(q):
    return H_MAT.T @ (R(q).T @ L(q)) @ H_MAT

def expq(phi):
    phi_norm = np.linalg.norm(phi)
    if phi_norm < 1e-10:
        return np.array([1.0, 0.0, 0.0, 0.0])
    return np.concatenate(([np.cos(phi_norm)], phi * (np.sin(phi_norm) / phi_norm)))

# --- FILTER INITIALIZATION ---
h = 0.05 
x = np.array([1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]) # [q, bias]
P = np.eye(6) * 0.5
V = np.diag(np.concatenate([3.8e-4 * np.ones(3), 1.8e-6 * np.ones(3)]))
W = 1e-2 * np.eye(3)
r_N = np.array([0, 0, 1.0]) 

# --- GUIDANCE CONFIG ---
SERIAL_PORT = '/dev/cu.usbmodem1101'
ser = serial.Serial(SERIAL_PORT, 115200, timeout=1)
CENTER_X, CENTER_Y = 160, 120
last_speak_time = time.time()

# STRICT PARAMETERS
STRICT_TOLERANCE = 10  # Tightened from 25 to 10 pixels
OMEGA_THRESHOLD = 1.2  # Slightly more sensitive "fast movement" check
STABILITY_REQ = 0.3    # Maximum omega allowed for "Centered" confirmation

print(f"--- STRICT MEKF System Active on {SERIAL_PORT} ---")

while True:
    try:
        line = ser.readline().decode().strip()
        if not line: continue
        vals = line.split(",")
        if len(vals) != 10: continue

        # 1. Parse Data
        cam_x, cam_y = int(vals[0]), int(vals[1])
        gyro = np.array([float(vals[4]), float(vals[5]), float(vals[6])])
        accel = np.array([float(vals[7]), float(vals[8]), float(vals[9])])
        
        a_norm = np.linalg.norm(accel)
        if a_norm < 1e-6: continue
        y_meas = accel / a_norm

        # --- 2. MEKF UPDATE ---
        q_old, beta = x[:4], x[4:7]
        omega_filt = gyro - beta #
        dq = expq(0.5 * h * omega_filt)
        
        q_pred = L(q_old) @ dq
        q_pred /= np.linalg.norm(q_pred)
        
        A = np.eye(6)
        A[:3, :3] = G(q_pred).T @ R(dq) @ G(q_old)
        A[:3, 3:6] = -0.5 * h * G(q_pred).T @ G(q_old)
        P_pred = A @ P @ A.T + V

        z = y_meas - (Q_mat(q_pred).T @ r_N)
        C = np.zeros((3, 6))
        C[:, :3] = H_MAT.T @ (L(q_pred).T @ L(H_MAT @ r_N) + R(q_pred) @ R(H_MAT @ r_N) @ T_MAT) @ G(q_pred)
        
        S = C @ P_pred @ C.T + W
        K = P_pred @ C.T @ np.linalg.inv(S)
        
        delta_x = K @ z
        phi = delta_x[:3]
        phi_sq = np.dot(phi, phi)
        dq_up = np.concatenate(([np.sqrt(max(0, 1-phi_sq)) if phi_sq < 1 else 0], phi))
        
        x[:4] = (L(q_pred) @ dq_up) / np.linalg.norm(L(q_pred) @ dq_up)
        x[4:7] = beta + delta_x[3:6]
        P = (np.eye(6) - K @ C) @ P_pred @ (np.eye(6) - K @ C).T + K @ W @ K.T

        # --- 3. TELEMETRY OUTPUT ---
        error_x = cam_x - CENTER_X
        error_y = cam_y - CENTER_Y
        omega_mag = np.linalg.norm(omega_filt)
        
        print("\n" + "="*40)
        print(f"CAMERA  | Err X: {error_x:3d} | Err Y: {error_y:3d}")
        print(f"MEKF BIAS| X: {x[4]:.3f} | Y: {x[5]:.3f} | Z: {x[6]:.3f}")
        print(f"STABILITY| |ω|: {omega_mag:.3f} rad/s")
        print("="*40)

        # --- 4. STRICT GUIDANCE LOGIC ---
        current_time = time.time()
        if current_time - last_speak_time > 3.5: # Faster feedback loop
            
            if omega_mag > OMEGA_THRESHOLD:
                text = "You're moving fast, let's slow down."
            
            # Strict Center: Must be within 10 pixels AND relatively still
            elif abs(error_x) < STRICT_TOLERANCE and abs(error_y) < STRICT_TOLERANCE:
                if omega_mag < STABILITY_REQ:
                    text = "You're centered. Proceed safely!"
                else:
                    text = "You're right there, just hold steady now."
            
            # Directional Cues with smaller deadbands
            elif error_x > STRICT_TOLERANCE:
                text = "Gently move to your right."
            elif error_x < -STRICT_TOLERANCE:
                text = "Move left."
            elif error_y > STRICT_TOLERANCE:
                text = "Tilt down."
            elif error_y < -STRICT_TOLERANCE:
                text = "Tilt up slightly."
            
            print(f"\n>>> VOICE: {text}\n")
            os.system(f'say "{text}"')
            last_speak_time = current_time

    except Exception as e:
        print(f"Stability Adjustment: {e}")
        P = np.eye(6) * 0.5

In [ ]:
import numpy as np
import serial
import time
import os

# --- MEKF CONSTANTS & UTILITIES ---
H_MAT = np.vstack([np.zeros((1, 3)), np.eye(3)])
T_MAT = np.block([[1, np.zeros((1, 3))], [np.zeros((3, 1)), -np.eye(3)]])

def hat(v):
    return np.array([[0, -v[2], v[1]], [v[2], 0, -v[0]], [-v[1], v[0], 0]])

def L(q):
    s, v = q[0], q[1:]
    m = np.zeros((4, 4))
    m[0, 0], m[0, 1:], m[1:, 0] = s, -v, v
    m[1:, 1:] = s * np.eye(3) + hat(v)
    return m

def R(q):
    s, v = q[0], q[1:]
    m = np.zeros((4, 4))
    m[0, 0], m[0, 1:], m[1:, 0] = s, -v, v
    m[1:, 1:] = s * np.eye(3) - hat(v)
    return m

def G(q):
    return L(q) @ H_MAT

def Q_mat(q):
    return H_MAT.T @ (R(q).T @ L(q)) @ H_MAT

def expq(phi):
    phi_norm = np.linalg.norm(phi)
    if phi_norm < 1e-10:
        return np.array([1.0, 0.0, 0.0, 0.0])
    return np.concatenate(([np.cos(phi_norm)], phi * (np.sin(phi_norm) / phi_norm)))

# --- FILTER INITIALIZATION ---
h = 0.05 
x = np.array([1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]) # [q, bias]
P = np.eye(6) * 0.5
V = np.diag(np.concatenate([3.8e-4 * np.ones(3), 1.8e-6 * np.ones(3)]))
W = 1e-2 * np.eye(3)
r_N = np.array([0, 0, 1.0]) 

# --- GUIDANCE CONFIG ---
SERIAL_PORT = '/dev/cu.usbmodem1101'
ser = serial.Serial(SERIAL_PORT, 115200, timeout=1)
CENTER_X, CENTER_Y = 160, 120
last_speak_time = time.time()

# STRICT PARAMETERS
STRICT_TOLERANCE = 10  
OMEGA_THRESHOLD = 1.2  
STABILITY_REQ = 0.3    

print(f"--- STRICT MEKF System Active on {SERIAL_PORT} ---")

while True:
    try:
        line = ser.readline().decode().strip()
        if not line: continue
        vals = line.split(",")
        if len(vals) != 10: continue

        # 1. Parse Data
        cam_x, cam_y = int(vals[0]), int(vals[1])
        gyro = np.array([float(vals[4]), float(vals[5]), float(vals[6])])
        accel = np.array([float(vals[7]), float(vals[8]), float(vals[9])])
        
        a_norm = np.linalg.norm(accel)
        if a_norm < 1e-6: continue
        y_meas = accel / a_norm

        # --- 2. MEKF UPDATE ---
        q_old, beta = x[:4], x[4:7]
        omega_filt = gyro - beta
        dq = expq(0.5 * h * omega_filt)
        
        q_pred = L(q_old) @ dq
        q_pred /= np.linalg.norm(q_pred)
        
        A = np.eye(6)
        A[:3, :3] = G(q_pred).T @ R(dq) @ G(q_old)
        A[:3, 3:6] = -0.5 * h * G(q_pred).T @ G(q_old)
        P_pred = A @ P @ A.T + V

        z = y_meas - (Q_mat(q_pred).T @ r_N)
        C = np.zeros((3, 6))
        C[:, :3] = H_MAT.T @ (L(q_pred).T @ L(H_MAT @ r_N) + R(q_pred) @ R(H_MAT @ r_N) @ T_MAT) @ G(q_pred)
        
        S = C @ P_pred @ C.T + W
        K = P_pred @ C.T @ np.linalg.inv(S)
        
        delta_x = K @ z
        phi = delta_x[:3]
        phi_sq = np.dot(phi, phi)
        dq_up = np.concatenate(([np.sqrt(max(0, 1-phi_sq)) if phi_sq < 1 else 0], phi))
        
        x[:4] = (L(q_pred) @ dq_up) / np.linalg.norm(L(q_pred) @ dq_up)
        x[4:7] = beta + delta_x[3:6]
        P = (np.eye(6) - K @ C) @ P_pred @ (np.eye(6) - K @ C).T + K @ W @ K.T

        # --- 3. DASHBOARD TELEMETRY ---
        error_x = cam_x - CENTER_X
        error_y = cam_y - CENTER_Y
        omega_mag = np.linalg.norm(omega_filt)
        
        print("\n" + "="*45)
        print(f"CAMERA  | Offset X: {error_x:4d} | Offset Y: {error_y:4d}")
        print(f"MEKF BIAS| X: {x[4]:6.3f} | Y: {x[5]:6.3f} | Z: {x[6]:6.3f}")
        print(f"DYNAMICS| |ω|: {omega_mag:6.3f} rad/s")
        print(f"STATUS  | {'STABLE' if omega_mag < STABILITY_REQ else 'MOVING'}")
        print("="*45)

        # --- 4. STRICT GUIDANCE LOGIC ---
        current_time = time.time()
        if current_time - last_speak_time > 3.5:
            
            if omega_mag > OMEGA_THRESHOLD:
                text = "You're moving fast, let's slow down."
            
            elif abs(error_x) < STRICT_TOLERANCE and abs(error_y) < STRICT_TOLERANCE:
                if omega_mag < STABILITY_REQ:
                    text = "You're centered. Proceed safely."
                else:
                    text = "You're shaking, proceed steady now."
            
            elif error_x > STRICT_TOLERANCE:
                text = "Move to your right."
            elif error_x < -STRICT_TOLERANCE:
                text = "Move to your left now."
            elif error_y > STRICT_TOLERANCE:
                text = "Tilt down."
            elif error_y < -STRICT_TOLERANCE:
                text = "Tilt up."
            else:
                text = "You're very close, just a tiny adjustment."

            print(f"\n>>> VOICE: {text}\n")
            os.system(f'say "{text}"')
            last_speak_time = current_time

    except Exception as e:
        print(f"Stability Adjustment: {e}")
        P = np.eye(6) * 0.5

--- STRICT MEKF System Active on /dev/cu.usbmodem1101 ---

CAMERA  | Offset X: -160 | Offset Y: -120
MEKF BIAS| X: -0.007 | Y: -0.005 | Z: -0.000
DYNAMICS| |ω|:  0.037 rad/s
STATUS  | STABLE

CAMERA  | Offset X: -160 | Offset Y: -120
MEKF BIAS| X: -0.907 | Y: -0.611 | Z:  0.001
DYNAMICS| |ω|:  0.069 rad/s
STATUS  | STABLE

CAMERA  | Offset X: -160 | Offset Y: -120
MEKF BIAS| X: -2.221 | Y: -1.575 | Z:  0.001
DYNAMICS| |ω|:  1.118 rad/s
STATUS  | MOVING

CAMERA  | Offset X: -160 | Offset Y: -120
MEKF BIAS| X: -3.156 | Y: -2.194 | Z:  0.001
DYNAMICS| |ω|:  2.763 rad/s
STATUS  | MOVING

CAMERA  | Offset X: -160 | Offset Y: -120
MEKF BIAS| X: -3.326 | Y: -2.339 | Z: -0.009
DYNAMICS| |ω|:  4.129 rad/s
STATUS  | MOVING

CAMERA  | Offset X: -160 | Offset Y: -120
MEKF BIAS| X: -3.097 | Y: -2.212 | Z: -0.020
DYNAMICS| |ω|:  4.113 rad/s
STATUS  | MOVING

CAMERA  | Offset X: -160 | Offset Y: -120
MEKF BIAS| X: -2.767 | Y: -1.997 | Z: -0.012
DYNAMICS| |ω|:  3.839 rad/s
STATUS  | MOVING

CAMERA  | 